# 🚦 Traffic Signal Detection - YOLOv8 Training

Trains a YOLOv8 object-detection model on the Roboflow **Signal Detection** dataset
(`Green Light` / `Red Light`) for the ESP32-Cam traffic-light audio-signal project.

**Workflow**
1. Check GPU and install dependencies
2. Upload and prepare the dataset zip (`Signal Detection.v1i.yolov8.zip`)
3. Sanity-check the data
4. Train YOLOv8 (transfer learning + augmentation)
5. Evaluate on the held-out test set
6. Preview predictions
7. Download the trained model

**Before you start:** `Runtime > Change runtime type > T4 GPU` (or any available GPU), then run the cells top to bottom.

> **On the ~95% accuracy target:** this export has only 67 images total (48 train / 13 val / 6 test)
> across two visually distinct classes, so a high mAP50 is realistic with transfer learning. But with
> a 6-image test set, any single metric will be noisy, and several source photos are consecutive burst
> shots of the same physical lights, so train/val/test likely contain near-duplicate frames. Treat the
> reported number as encouraging rather than proof of real-world accuracy - for a sturdier model, add
> more varied images later (different lights, angles, lighting, times of day) and retrain.

## 1. Environment setup

Confirm a GPU is attached, then install `ultralytics` (includes YOLOv8).

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 2. Upload and prepare the dataset

Run the next cell, then click **Choose Files** and select your Roboflow export
(`Signal Detection.v1i.yolov8.zip`). Upload the **zip file** itself, not an extracted folder.

In [ ]:
from google.colab import files

print("Select your dataset zip file (e.g. 'Signal Detection.v1i.yolov8.zip')")
uploaded = files.upload()
if len(uploaded) != 1:
    print(f"Warning: expected 1 file, got {len(uploaded)}. Using the first one.")
zip_name = next(iter(uploaded))
print(f"\nUploaded '{zip_name}' ({len(uploaded[zip_name]) / 1e6:.1f} MB)")

In [ ]:
import glob
import shutil
import zipfile
from pathlib import Path

import yaml

DATASET_DIR = Path("/content/dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

with zipfile.ZipFile(f"/content/{zip_name}", "r") as zf:
    zf.extractall(DATASET_DIR)

# The zip may extract flat (data.yaml at the root) or nested inside a folder -
# search for it so this works either way.
yaml_matches = glob.glob(str(DATASET_DIR / "**" / "data.yaml"), recursive=True)
assert yaml_matches, f"No data.yaml found inside {zip_name} - check it's the Roboflow YOLOv8 export."
DATA_YAML = Path(yaml_matches[0])
DATASET_ROOT = DATA_YAML.parent
print("Dataset root:", DATASET_ROOT)

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

# Roboflow's data.yaml uses relative paths like '../train/images' that assume a
# specific nesting. Rewrite them as absolute paths so training works regardless
# of how the zip happened to be structured.
split_folders = {"train": "train", "val": "valid", "test": "test"}
for split_key, folder in split_folders.items():
    img_dir = DATASET_ROOT / folder / "images"
    if img_dir.exists():
        data_cfg[split_key] = str(img_dir)

with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f, sort_keys=False)

print(yaml.dump(data_cfg, sort_keys=False))

In [ ]:
print(f"{'split':6s}  images  labels")
for folder in ["train", "valid", "test"]:
    img_dir = DATASET_ROOT / folder / "images"
    lbl_dir = DATASET_ROOT / folder / "labels"
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"{folder:6s}  {n_img:6d}  {n_lbl:6d}")

print("\nClasses:", data_cfg["names"])

### Preview a few labeled samples

The Roboflow annotations are stored as polygons (one line per object: `class x1 y1 x2 y2 ...`).
YOLOv8's detector automatically derives a bounding box from each polygon during training, so this
preview draws the tight box around each one as a sanity check.

In [ ]:
import random

import cv2
import matplotlib.pyplot as plt

%matplotlib inline

CLASS_COLORS = {0: (0, 200, 0), 1: (220, 0, 0)}  # Green Light, Red Light (RGB)

def draw_labels(img_path, lbl_path, class_names):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            vals = line.split()
            cls_id = int(vals[0])
            coords = [float(v) for v in vals[1:]]
            xs = [coords[i] * w for i in range(0, len(coords), 2)]
            ys = [coords[i] * h for i in range(1, len(coords), 2)]
            x1, y1, x2, y2 = int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))
            color = CLASS_COLORS.get(cls_id, (255, 255, 0))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, class_names[cls_id], (x1, max(y1 - 6, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img

train_images = sorted((DATASET_ROOT / "train" / "images").glob("*"))
sample = random.sample(train_images, min(6, len(train_images)))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, img_path in zip(axes.ravel(), sample):
    lbl_path = DATASET_ROOT / "train" / "labels" / f"{img_path.stem}.txt"
    ax.imshow(draw_labels(img_path, lbl_path, data_cfg["names"]))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Train YOLOv8

Choices made for this specific dataset:
- **Transfer learning** from COCO-pretrained weights - essential with only 48 training images.
- **`yolov8s`** as the default size: enough capacity to separate two simple color classes without
  overfitting as easily as `m`/`l` would on this little data. Change `MODEL_SIZE` to `yolov8n` (faster)
  or `yolov8m` (more capacity, more overfit risk) to compare.
- **Augmentation** (rotation, translation, scale, shear, HSV jitter, mosaic, light mixup, horizontal
  flip) to manufacture visual variety from a small set. Vertical flip is disabled since a photographed
  signal is never upside-down.
- **Early stopping** (`patience`) halts training once validation performance plateaus, which also
  guards against overfitting.
- These same polygon labels also work unchanged for instance segmentation - swap in a `yolov8s-seg.pt`
  checkpoint if you ever need pixel masks instead of boxes.

Typically finishes in well under 30 minutes on a free Colab GPU, given how small this dataset is.

In [ ]:
import torch
from ultralytics import YOLO

MODEL_SIZE = "yolov8s"    # yolov8n (fastest) | yolov8s (default) | yolov8m (more capacity)
EPOCHS = 150
IMGSZ = 640                # source images are 512x512; 640 matches the COCO-pretrained checkpoints
BATCH = 16
PATIENCE = 50               # stop early if val performance plateaus for this many epochs
RUN_NAME = "signal_detection"

device = 0 if torch.cuda.is_available() else "cpu"
print("Training device:", "GPU" if device == 0 else "CPU (enable a GPU runtime for reasonable speed)")

model = YOLO(f"{MODEL_SIZE}.pt")

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=device,
    project="runs_signal_detection",
    name=RUN_NAME,
    seed=42,
    cos_lr=True,
    close_mosaic=10,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    mixup=0.1,
    plots=True,
)

RUN_DIR = Path(model.trainer.save_dir)
print("\nRun directory:", RUN_DIR)

## 4. Evaluate on the test set

Object detection doesn't have a single "accuracy" figure - the standard metrics are **mAP50** (mean
average precision at 0.5 IoU, the closest analog to accuracy), **mAP50-95**, **precision**, and
**recall**. With only 6 test images, treat these as directional rather than statistically solid.

In [ ]:
best_model = YOLO(RUN_DIR / "weights" / "best.pt")
test_metrics = best_model.val(data=str(DATA_YAML), split="test")

print("Test set metrics:")
for key, value in test_metrics.results_dict.items():
    print(f"  {key:<28}{value:.3f}")

try:
    print("\nPer-class mAP50:")
    for i, cls_idx in enumerate(test_metrics.box.ap_class_index):
        print(f"  {test_metrics.names[cls_idx]:<15}{test_metrics.box.ap50[i]:.3f}")
except Exception as e:
    print(f"(per-class breakdown unavailable: {e})")

map50 = test_metrics.results_dict.get("metrics/mAP50(B)", 0.0)
target = 0.95
verdict = "reached" if map50 >= target else "below"
print(f"\nmAP50 = {map50:.1%} ({verdict} the {target:.0%} target - the closest detection analog to accuracy)")

### Training curves and confusion matrix

Ultralytics automatically saves diagnostic plots to the run directory during training/validation.

In [ ]:
from IPython.display import Image, display

for plot_name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        print(plot_name)
        display(Image(filename=str(plot_path), width=700))

## 5. Preview predictions on test images

In [ ]:
test_images = sorted((DATASET_ROOT / "test" / "images").glob("*"))
preds = best_model.predict(source=[str(p) for p in test_images], conf=0.25, save=False)

n = len(preds)
cols = min(3, n) if n else 1
rows = (n + cols - 1) // cols if n else 1
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
axes = [axes] if n <= 1 else axes.ravel()
for ax, pred in zip(axes, preds):
    ax.imshow(cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(Path(pred.path).name, fontsize=8)
    ax.axis("off")
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Download the trained model

Packages the best weights, last checkpoint, and all training/validation plots into a zip, then
triggers browser downloads. If your browser blocks the second download, allow multiple downloads
for this site and re-run the cell.

In [ ]:
import shutil

from google.colab import files

archive_path = shutil.make_archive(f"/content/{RUN_NAME}_results", "zip", RUN_DIR)
print(f"Packaged run folder: {archive_path} ({Path(archive_path).stat().st_size / 1e6:.1f} MB)")

files.download(str(RUN_DIR / "weights" / "best.pt"))
files.download(archive_path)

### Optional: export for deployment

`best.pt` is the standard PyTorch checkpoint. If you plan to run inference outside PyTorch (for
example, a lightweight server process feeding the ESP32-Cam pipeline), export to ONNX instead.

In [ ]:
onnx_path = best_model.export(format="onnx", imgsz=IMGSZ, simplify=True)

from google.colab import files
files.download(str(onnx_path))

## Done

You now have:
- `best.pt` - the trained YOLOv8 weights (best validation performance during the run)
- `<run_name>_results.zip` - weights + training curves + confusion matrix
- optionally, an ONNX export

To reuse the model later:
```python
from ultralytics import YOLO
model = YOLO("best.pt")
results = model.predict("image.jpg")
```